# Indraneel - LSTM training on Colab

This notebook clones the Indraneel branch, installs the repository's pinned environment, prepares UCI HAR through `src/data.py`, runs `src/train_lstm.py`, and downloads the saved model and results. Training code stays reusable in the Python script.

The local branch must be pushed before cloning: `git push -u origin indraneel-app`.

## 1. Select a GPU runtime

In Colab, choose **Runtime > Change runtime type > T4 GPU** (or another available GPU). The project uses TensorFlow 2.16.1 from `requirements.txt`; if Colab asks for a runtime restart after installation, restart and rerun from the clone step.

In [ ]:
import os
import subprocess
import sys

REPO_URL = 'https://github.com/aryan-2206/Human_Activity_Recognition.git'
BRANCH = 'indraneel-app'
REPO_DIR = '/content/Human_Activity_Recognition'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', BRANCH], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Working directory:', os.getcwd())

## 2. Install dependencies and verify the accelerator

Keep TensorFlow and scikit-learn aligned with the project environment so saved models load consistently.

In [ ]:
%pip install -r requirements.txt
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))

## 3. Prepare and inspect the shared data

`prepare_data()` downloads UCI HAR if needed, applies the train-fitted channel scaler, and writes the shared windows, labels, subjects, and label names under `data/processed/`. The generated files are ignored by Git, so they are created in this Colab clone.

In [ ]:
from src.data import prepare_data

X_train, X_test, y_train, y_test, subjects_train, subjects_test = prepare_data()
print('Train:', X_train.shape, y_train.shape, subjects_train.shape)
print('Test: ', X_test.shape, y_test.shape, subjects_test.shape)

In [ ]:
import json
import numpy as np
from pathlib import Path

label_names = json.loads(Path('data/processed/label_names.json').read_text())
assert X_train.ndim == 3 and X_train.shape[1:] == (128, 6)
assert X_test.ndim == 3 and X_test.shape[1:] == (128, 6)
assert len(X_train) == len(y_train) == len(subjects_train)
assert len(X_test) == len(y_test) == len(subjects_test)
print('Labels:', label_names)
print('Training subjects:', np.unique(subjects_train).size)

## 4. Train and evaluate the LSTM

The script holds out about 15% of training subjects for validation, trains the two-layer LSTM with gradient clipping, early stopping, and learning-rate reduction, then evaluates on the official test split. It saves `models/lstm.keras`, `results/lstm_metrics.json`, and `results/lstm_curves.png`. The metric JSON includes the shared helper output plus `macro_f1` and `per_class_report` aliases required by the original common prompt.

In [ ]:
!python src/train_lstm.py --epochs 40 --batch-size 128 --validation-fraction 0.15 --seed 42

## 5. Download artifacts and return them to the branch

Download these outputs and copy them into the matching paths in your local `indraneel-app` checkout. Keep files over about 50 MB on Drive and add a README link. Remove this notebook manually if the team chooses not to retain it in GitHub.

In [ ]:
from pathlib import Path
from google.colab import files

output_paths = [
    Path('models/lstm.keras'),
    Path('results/lstm_metrics.json'),
    Path('results/lstm_curves.png'),
]
missing = [str(path) for path in output_paths if not path.is_file()]
if missing:
    raise FileNotFoundError(f'Missing expected training outputs: {missing}')
for path in output_paths:
    files.download(str(path))